In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
from pathlib import Path
from pprint import pprint
import gc
from sklearn.model_selection import train_test_split

In [3]:
!pip install catboost
import catboost

^C


ModuleNotFoundError: No module named 'catboost'

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 1.0/100.2 MB 3.9 MB/s eta 0:00:26
    --------------------------------------- 1.6/100.2 MB 3.5 MB/s eta 0:00:29
    --------------------------------------- 2.4/100.2 MB 3.1 MB/s eta 0:00:32
   - -------------------------------------- 2.6/100.2 MB 3.0 MB/s eta 0:00:33
   - -------------------------------------- 3.1/100.2 MB 2.6 MB/s eta 0:00:37
   - -------------------------------------- 3.4/100.2 MB 2.5 MB/s eta 0:00:39
   - -------------------------------------- 3.7/100.2 MB 2.5 MB/s eta 0:00:40
   - -------------------------------------- 4.2/100.2 MB 2.4 MB/s eta 0:00:41
   - -------------------------------------- 4.5/100.2 MB 2.3 MB/s eta 0:00:42
   - -------------------------------------- 4.7/100.2 MB 2.2 MB/s eta 0:00:43
   - -------------------------------------- 5.0/100.2 MB 2.1 MB/s eta 0:00:45


In [7]:
!pip install xgboost
import xgboost

In [2]:
!pip install sentence-transformers

In [5]:
'''Эмбеддинг от OpenAI стоит денег. (5$)'''
from openai import OpenAI
import os

<h1>Эмбеддинги для категориальных признаков</h1>
Категориальные признаки можно эмбеддить по-разному.
На базовом уровне можно использовать one-hot кодирование.
Более сложная альтернатива - предобученные эмбеддеры, один из которых представлен ниже.

In [4]:
from sentence_transformers import SentenceTransformer
texts = ['Where is Gamora?', 'Who is Gamora?', 'Why is Gamora?']
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\andra\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andra\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
model.encode(texts)

array([[ 0.05880961, -0.06816845, -0.02992233, ...,  0.02165488,
        -0.3030633 , -0.05126124],
       [-0.3230413 ,  0.06128918, -0.15995   , ..., -0.06036601,
        -0.05101775,  0.01018316],
       [-0.21942565,  0.00095907, -0.14654884, ...,  0.13390395,
         0.3018818 , -0.02924964]], shape=(3, 384), dtype=float32)

In [7]:
ml_path = Path.cwd()
base_path = ml_path.parent
data_path = base_path / 'data_preparation' / 'data'
clean_data_path = data_path / 'clean'
clean_data_file = clean_data_path / 'clean_data.xlsx'

In [8]:
CLEAN_DATA = pd.read_excel(clean_data_file)

In [36]:
TRIMMED_DATA = CLEAN_DATA.drop(['lead_tags', 'contact_Город', 'contact_id', 'lead_id', 'lead_Состав заказа', 'lead_yclid'], axis = 1)

In [13]:
TRIMMED_DATA.columns.tolist()

['sale_ts',
 'sale_date',
 'buyout_flag',
 'handed_to_delivery_ts',
 'issued_or_pvz_ts',
 'days_to_outcome',
 'lead_price',
 'lead_responsible_user_id',
 'lead_Служба доставки',
 'lead_Вид оплаты',
 'lead_Проблема',
 'lead_Оплата МОП',
 'lead_Дата получения денег на Р/С',
 'contact_LTV',
 'contact_Число сделок',
 'lead_Источник',
 'lead_Условный отказ',
 'lead_Тариф Доставки',
 'lead_Скидка',
 'lead_Квалификация лида',
 'lead_Дата создания сделки',
 'lead_Дата перехода в Сборку',
 'has_yclid',
 'is_paid_mop',
 'is_repeat_client',
 'has_promo',
 'lead_source_category',
 'lead_quality',
 'is_yur',
 'product_category',
 'has_маска',
 'has_наколенник',
 'has_бандаж_шейный',
 'has_повязка',
 'has_напульсник',
 'has_обувь',
 'has_подушка',
 'has_матрас',
 'has_пояс',
 'has_аксессуары',
 'has_крем',
 'n_product_categories']

In [37]:
TRIMMED_DATA = TRIMMED_DATA.drop(['closed_ts', 'received_ts', 'rejected_ts', 'returned_ts', 'days_to_outcome', 'lead_Условный отказ', 
                                 'lead_Оплата МОП', 'is_paid_mop', 'lead_Дата получения денег на Р/С'], axis = 1)

In [60]:
TRIMMED_DATA.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17966 entries, 0 to 17965
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   sale_ts                      17966 non-null  datetime64[ns]
 1   sale_date                    17966 non-null  datetime64[ns]
 2   buyout_flag                  17966 non-null  int64         
 3   handed_to_delivery_ts        17541 non-null  datetime64[ns]
 4   issued_or_pvz_ts             17206 non-null  datetime64[ns]
 5   lead_price                   17966 non-null  int64         
 6   lead_responsible_user_id     17966 non-null  object        
 7   lead_Служба доставки         17951 non-null  object        
 8   lead_Вид оплаты              17944 non-null  object        
 9   lead_Проблема                17939 non-null  object        
 10  contact_LTV                  13896 non-null  float64       
 11  contact_Число сделок         13896 non-nu

In [79]:
def sig3OutlDetector(DATA, columns):
    Stds = DATA[columns].std()
    Means = DATA[columns].mean()
    CrvdDATA = DATA[columns]
    CrvdDATA = CrvdDATA[((-3*Stds+Means<=CrvdDATA)&(3*Stds+Means>=CrvdDATA))[columns[0]]]
    return DATA.iloc[CrvdDATA.index.tolist()].reset_index(drop = True)
def bool2flag(DATA):
    DAT = DATA.copy()
    cols = DAT.select_dtypes(['bool']).columns.tolist()
    DAT[cols] = DAT[cols]*1
    return DAT
def Nan2Missing(DATA):
    DAT = DATA.copy()
    cols = DAT.select_dtypes(['object']).columns.tolist()
    DAT[cols] = DAT[cols].fillna('ПРОПУСК')
    return DAT
def Nan2Adequate(DATA):
    pass

In [85]:
EMBEDDED_DATA = bool2flag(TRIMMED_DATA)

In [86]:
EMBEDDED_DATA = sig3OutlDetector(EMBEDDED_DATA, ['lead_price'])

In [82]:
EMBEDDED_DATA = Nan2Missing(EMBEDDED_DATA)

In [87]:
Y = EMBEDDED_DATA['buyout_flag']
X = EMBEDDED_DATA.drop('buyout_flag', axis = 1)

In [88]:
X.isna().sum()

sale_ts                            0
sale_date                          0
handed_to_delivery_ts            418
issued_or_pvz_ts                 746
lead_price                         0
lead_responsible_user_id           0
lead_Служба доставки              15
lead_Вид оплаты                   22
lead_Проблема                     27
contact_LTV                     4027
contact_Число сделок            4027
lead_Источник                  13282
lead_Тариф Доставки             1060
lead_Скидка                    15249
lead_Квалификация лида          6461
lead_Дата создания сделки      10377
lead_Дата перехода в Сборку     9522
has_yclid                          0
is_repeat_client                   0
has_promo                          0
lead_source_category            1443
lead_quality                   14678
is_yur                             0
product_category                  22
has_маска                          0
has_наколенник                     0
has_бандаж_шейный                  0
h

In [ ]:
depth = 10
trees = 5000
lr = 1e-2
earStop = 20
seed = 42
classes = 3

In [ ]:
Version = 'MK1'

In [ ]:
XGB_Classifier = xgb.XGBClassifier(
    max_depth = depth,
    learning_rate = lr,
    n_estimators = trees,
    objective = 'binary:logistic',
    num_class = classes,
    eval_metric = 'auc',
    early_stopping_rounds=earStop,
    seed = seed
)

In [ ]:
CB_Classifier = cb.CatBoostClassifier(
    num_trees = trees,
    depth = depth,
    learning_rate = lr,
    classes_count = classes,
    loss_function = 'AUC',
    early_stopping_rounds=earStop
)

In [ ]:
from datetime import datetime
xgbStart = 0
xgbEnd = 0
cbStart = 0
cbEnd = 0

In [ ]:
xgbStart = datetime.now()
XGB_Classifier.fit(
    TrainDataset, TrainLabels,
    eval_set = [(TrainDataset, TrainLabels), (TestDataset, TestLabels)],
    verbose = True
)
xgbEnd = datetime.now()
xgbBestTime = str(xgbEnd-xgbStart).split('.')[0]
xgbTotalTime = xgbBestTime

In [ ]:
cbStart = datetime.now()
CB_Classifier.fit(
    TrainDataset, TrainLabels,
    eval_set = (TestDataset, TestLabels),
    verbose = True
)
cbEnd = datetime.now()
cbBestTime = str(cbEnd-cbStart).split('.')[0]
cbTotalTime = cbBestTime

In [ ]:
x_log_res = XGB_Classifier.evals_result()
x_log_keys = [i for i in x_log_res.keys()]
x_log_metrics = [i for i in x_log_res[x_log_keys[0]].keys()]
x_epochs = len(x_log_res[x_log_keys[0]][x_log_metrics[0]])
x_Epoch_range = np.arange(x_epochs, dtype=np.int64)
x_Train_metrics = np.array(x_log_res[x_log_keys[0]][x_log_metrics[0]])
x_Test_metrics = np.array(x_log_res[x_log_keys[1]][x_log_metrics[0]])
tmp = ['train', 'validation']
x_Keys = {i: j for i, j in zip(x_log_keys, tmp)}
tmp = ['AUC']
x_Metrics = {i: j for i, j in zip(x_log_metrics, tmp)}

In [ ]:
c_log_res = CB_Classifier.get_evals_result()
c_log_keys = [i for i in c_log_res.keys()]
c_log_metrics = [i for i in c_log_res[c_log_keys[0]].keys()]
c_epochs = len(c_log_res[c_log_keys[0]][c_log_metrics[0]])
c_Epoch_range = np.arange(c_epochs, dtype=np.int64)
c_Train_metrics = np.array(c_log_res[c_log_keys[0]][c_log_metrics[0]])
c_Test_metrics = np.array(c_log_res[c_log_keys[1]][c_log_metrics[0]])
tmp = ['train', 'validation']
c_Keys = {i: j for i, j in zip(c_log_keys, tmp)}
tmp = ['~AUC']
c_Metrics = {i: j for i, j in zip(c_log_metrics, tmp)}

In [ ]:
log_path = ml_path / 'Logs'
image_path = ml_path / 'Images'

In [ ]:
def VisualizeLogsWMinVal(Epoch_range, Train_metrics, Test_metrics, Keys, Metrics, log_keys, log_metrics, model, kernel=None, save=False, log_path = '', image_path = '', title='1'):
  fig, ax = plt.subplots(len(Metrics), figsize = (8, 4*len(Metrics)))
  font = {'font.size': 13}
  plt.rcParams.update(font)
  if len(Metrics) == 1:
    Min_metric = Test_metrics.min()
    Min_epoch = Epoch_range[np.argmin(Test_metrics)]
    ax.plot(Epoch_range, Train_metrics, label = f'{Keys[log_keys[0]]} {Metrics[log_metrics[0]]}', zorder = 2)
    ax.plot(Epoch_range, Test_metrics, label = f'{Keys[log_keys[1]]} {Metrics[log_metrics[0]]}', zorder = 3)
    ax.scatter(Min_epoch, Min_metric, c = 'crimson', marker='s', s=20, label = f'Min val {Metrics[log_metrics[0]]} = {Min_metric:.3f}', zorder = 4)
    ax.plot([Min_epoch, Min_epoch], [0, Min_metric], c = 'crimson', linestyle = '--')
    ax.legend(loc=0)
    ax.set_xlabel('Epochs')
    ax.set_ylabel(Metrics[log_metrics[0]])
    if kernel is not None:
      ax.set_title(f'Logging of {model} training (kernel = {kernel})')
    else:
      ax.set_title(f'Logging of {model} training')
    ax.grid(True, zorder = 1)
  else:
    if kernel is not None:
      ax[0].set_title(f'Logging of {model} training (kernel = {kernel})')
    else:
      ax[0].set_title(f'Logging of {model} training')
    for i, metric in enumerate(log_metrics):
      Min_epoch = Epoch_range[np.argmin(Test_metrics[metric])]
      Min_metric = Test_metrics[metric][Min_epoch]
      ax[i].plot(Epoch_range, Train_metrics[metric], label = f'{Keys[log_keys[0]]} {Metrics[metric]}', zorder = 2)
      ax[i].plot(Epoch_range, Test_metrics[metric], label = f'{Keys[log_keys[1]]} {Metrics[metric]}', zorder = 3)
      ax[i].scatter(Min_epoch, Min_metric, c = 'crimson', marker='s', s=20, label = f'Min val {Metrics[metric]} = {Min_metric:.3f}', zorder = 4)
      ax[i].plot([Min_epoch, Min_epoch], [0, Min_metric], c = 'crimson', linestyle = '--')
      ax[i].legend(loc=0)
      ax[i].set_xlabel('Epochs')
      ax[i].set_ylabel(Metrics[metric])
      ax[i].grid(True, zorder = 1)
  if save:
    Dict = {'Epochs': Epoch_range}
    if len(Metrics) == 1:
      Dict[f'{Keys[log_keys[0]]}_{Metrics[log_metrics[0]]}'] = Train_metrics
      Dict[f'{Keys[log_keys[1]]}_{Metrics[log_metrics[0]]}'] = Test_metrics
    else:
      for metric in log_metrics:
        Dict[f'{Keys[log_keys[0]]}_{Metrics[metric]}'] = Train_metrics[metric]
        Dict[f'{Keys[log_keys[1]]}_{Metrics[metric]}'] = Test_metrics[metric]
    df = pd.DataFrame(Dict)
    df.to_csv(os.path.join(log_path, title+'.csv'))
    fig.savefig(os.path.join(image_path, title+'.png'))
  plt.show()

In [ ]:
VisualizeLogsWMinVal(x_Epoch_range, x_Train_metrics, x_Test_metrics, x_Keys, x_Metrics,
                     x_log_keys, x_log_metrics, model = 'XGB', kernel=kernel, save=False,
                     log_path=log_path, image_path=image_path,
                     title = f'XGB-{Version}_log_{kernel}_{trees}')